In [60]:
import pandas as pd
import numpy as np

from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

In [34]:
books_features = pd.read_csv("../Data/books_features.csv")
ratings = pd.read_csv("../Data/ratings.csv")

books_features.head()
ratings.head()

,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3


In [35]:
ratings["rating"].value_counts().sort_index()

rating
1     124195
2     359257
3    1370916
4    2139018
5    1983093
Name: count, dtype: int64

In [36]:
relevant_ratings = ratings[ratings["rating"] >= 4].copy()

relevant_ratings.head()

,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
5,2,26,4


In [37]:
relevant_ratings.shape

(4122111, 3)

In [38]:
relevant_ratings["user_id"].nunique(), relevant_ratings["book_id"].nunique()

(53406, 10000)

In [39]:
relevant_counts = (
    relevant_ratings
    .groupby("user_id")
    .size()
)

relevant_counts.describe()

count    53406.000000
mean        77.184417
std         26.038128
min          1.000000
25%         60.000000
50%         76.000000
75%         93.000000
max        194.000000
dtype: float64

In [40]:
eligible_users = relevant_counts[relevant_counts >= 2]

eligible_users.shape

(53398,)

In [41]:
eligible_users.describe()

count    53398.000000
mean        77.195831
std         26.023374
min          2.000000
25%         60.000000
50%         76.000000
75%         93.000000
max        194.000000
dtype: float64

In [42]:
np.random.seed(42)

user_book_groups = (
    relevant_ratings
    .groupby("user_id")["book_id"]
    .apply(np.array)
)

evaluation_pairs = []

for user_id, user_books in user_book_groups.items():
    
    if len(user_books) < 2:
        continue
    
    test_book = np.random.choice(user_books)
    
    remaining_books = user_books[user_books != test_book]
    seed_book = np.random.choice(remaining_books)
    
    evaluation_pairs.append(
        [user_id, seed_book, test_book]
    )

evaluation_pairs = pd.DataFrame(
    evaluation_pairs,
    columns=["user_id", "seed_book_id", "test_book_id"]
)

evaluation_pairs.head()

,user_id,seed_book_id,test_book_id
0,1,6285,162
1,2,127,8855
2,3,4090,7401
3,4,3469,10
4,5,852,262


In [43]:
print("Seed books found:",
      evaluation_pairs["seed_book_id"].isin(
          books_features["book_id"]
      ).sum())

print("Test books found:",
      evaluation_pairs["test_book_id"].isin(
          books_features["book_id"]
      ).sum())

print("Total evaluation pairs:",
      len(evaluation_pairs))

Seed books found: 53398
Test books found: 53398
Total evaluation pairs: 53398


In [44]:
evaluation_sample = evaluation_pairs.sample(
    n=1000,
    random_state=42
).reset_index(drop=True)

evaluation_sample.head()

,user_id,seed_book_id,test_book_id
0,2976,856,1015
1,14501,144,6406
2,52153,4593,466
3,30203,3492,2086
4,25134,3504,1552


In [45]:
book_id_to_index = pd.Series(
    books_features.index,
    index=books_features["book_id"]
).to_dict()

len(book_id_to_index)

10000

In [46]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(
    books_features["tag_name"]
)

content_similarity = cosine_similarity(tfidf_matrix)

content_similarity.shape

(10000, 10000)

In [47]:
from scipy.sparse import csr_matrix

user_ids = ratings["user_id"].astype("category")
book_ids = ratings["book_id"].astype("category")

user_item_matrix = csr_matrix(
    (
        ratings["rating"].values,
        (user_ids.cat.codes, book_ids.cat.codes)
    )
)

user_item_matrix.shape

(53424, 10000)

In [48]:
collaborative_model = NearestNeighbors(
    n_neighbors=11,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

collaborative_model.fit(user_item_matrix.T)

print("Collaborative model fitted.")

Collaborative model fitted.


In [49]:
evaluation_ratings = ratings.merge(
    evaluation_sample[
        ["user_id", "test_book_id"]
    ],
    on="user_id",
    how="left"
)

evaluation_ratings = evaluation_ratings[
    ~(
        (evaluation_ratings["book_id"] == evaluation_ratings["test_book_id"])
        & evaluation_ratings["test_book_id"].notna()
    )
].copy()

evaluation_ratings = evaluation_ratings[
    ["user_id", "book_id", "rating"]
]

evaluation_ratings.shape

(5975479, 3)

In [50]:
eval_user_ids = evaluation_ratings["user_id"].astype("category")
eval_book_ids = evaluation_ratings["book_id"].astype("category")

eval_user_item_matrix = csr_matrix(
    (
        evaluation_ratings["rating"].values,
        (eval_user_ids.cat.codes, eval_book_ids.cat.codes)
    ),
    shape=(len(eval_user_ids.cat.categories), len(eval_book_ids.cat.categories))
)

eval_user_item_matrix.shape

(53424, 10000)

In [51]:
eval_collaborative_model = NearestNeighbors(
    n_neighbors=11,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

eval_collaborative_model.fit(
    eval_user_item_matrix.T
)

print("Evaluation collaborative model fitted.")

Evaluation collaborative model fitted.


In [52]:
test_case = evaluation_sample.iloc[0]

test_case

user_id         2976
seed_book_id     856
test_book_id    1015
Name: 0, dtype: int64

In [53]:
seed_book_id = test_case["seed_book_id"]
test_book_id = test_case["test_book_id"]

seed_book_id, test_book_id

(np.int64(856), np.int64(1015))

In [54]:
seed_book = books_features[
    books_features["book_id"] == seed_book_id
][["book_id", "title", "authors"]]

test_book = books_features[
    books_features["book_id"] == test_book_id
][["book_id", "title", "authors"]]

print("SEED BOOK")
display(seed_book)

print("\nTEST BOOK")
display(test_book)

SEED BOOK


,book_id,title,authors
855,856,"Gathering Blue (The Giver, #2)",Lois Lowry



TEST BOOK


,book_id,title,authors
1014,1015,Steppenwolf,"Hermann Hesse, Basil Creighton"


In [55]:
seed_title = books_features.loc[
    books_features["book_id"] == seed_book_id,
    "title"
].iloc[0]

seed_title

'Gathering Blue (The Giver, #2)'

In [56]:
def recommend_content_eval(book_id, n=10):
    
    # Find the book's position
    if book_id not in book_id_to_index:
        return pd.DataFrame()
    
    idx = book_id_to_index[book_id]
    
    # Get similarity scores
    scores = content_similarity[idx]
    
    # Sort by similarity
    top_indices = np.argsort(scores)[::-1]
    
    # Remove the original book
    top_indices = [
        i for i in top_indices
        if i != idx
    ][:n]
    
    recommendations = books_features.iloc[top_indices].copy()
    
    recommendations["similarity"] = scores[top_indices]
    
    return recommendations[
        ["book_id", "title", "authors", "similarity"]
    ]

In [57]:
content_recommendations = recommend_content_eval(
    seed_book_id,
    n=10
)

content_recommendations

,book_id,title,authors,similarity
1289,1290,"Messenger (The Giver, #3)",Lois Lowry,0.863327
485,486,"The City of Ember (Book of Ember, #1)",Jeanne DuPrau,0.791112
1902,1903,"Son (The Giver, #4)",Lois Lowry,0.789708
6163,6164,"Among the Betrayed (Shadow Children, #3)",Margaret Peterson Haddix,0.784124
5540,5541,"Among the Impostors (Shadow Children, #2)",Margaret Peterson Haddix,0.771667
8509,8510,"Among the Enemy (Shadow Children, #6)",Margaret Peterson Haddix,0.765565
3083,3084,"The People of Sparks (Book of Ember, #2)",Jeanne DuPrau,0.764836
7303,7304,"Among the Barons (Shadow Children, #4)",Margaret Peterson Haddix,0.763559
5889,5890,"The Prophet of Yonwood (Book of Ember, #3)",Jeanne DuPrau,0.757566
6900,6901,"The Diamond of Darkhold (Book of Ember, #4)",Jeanne DuPrau,0.742242


In [58]:
test_book_id in content_recommendations["book_id"].values

False

In [59]:
content_matches = content_recommendations[
    content_recommendations["book_id"] == test_book_id
]

content_matches

,book_id,title,authors,similarity


In [61]:
def get_content_recommendations(book_id, k=10):
    
    if book_id not in book_id_to_index:
        return []
    
    idx = book_id_to_index[book_id]
    
    scores = content_similarity[idx]
    
    top_indices = np.argsort(scores)[::-1]
    
    # Remove the seed book
    top_indices = top_indices[top_indices != idx][:k]
    
    return books_features.iloc[top_indices]["book_id"].tolist()

In [62]:
content_test_recommendations = get_content_recommendations(
    seed_book_id,
    k=10
)

content_test_recommendations

[1290, 486, 1903, 6164, 5541, 8510, 3084, 7304, 5890, 6901]

In [63]:
test_book_id in content_test_recommendations

False